## Introduction
This notebook details how to train a model using Google Colab or locally. When training locally, you will need access to an NVidia GPU or a considerable amount of patience. When training on Colab, use model_trainer.py instead of this notebook. The contents are the same, but instead of an interactive session through a notebook, all parameters are passed in directly through command line arguments. Alternatively, you can upload this to Colab so that it can be used in an interactive session.

*Note:* If you have an AMD GPU, you may wish to use ROCm to train models. Do so using the __model_trainer.py__, not this notebook. You will need to download the dataset *locally* instead of using RoboFlow, as the RoboFlow library has many dependency conflicts with PyTorch for ROCm.

This document will detail four sections:
- Installing dependencies: Not needed if running in a kernel that already has all the dependencies set up. Kernels should be created using virtual environments or Conda.
- Importing a dataset from RoboFlow: Will rely on several pieces of user input, including workspace, dataset name, API key, etc.
- Model training: Will be done through YOLO, starting at YOLOv11s, YOLOv11n, YOLOv26s, or YOLOv26n. The type of model will be set through user input.
    - Several options will be allowed for training purposes. These will include augmentations, number of epochs, and various important setup information:
        - Epochs (default 300)
        - Patience (default 50)
        - Rotate (defaults to disabled)
        - Shear (defaults to disabled)
        - Mosaic (defaults to disabled)
        - Flip L/R (defaults to disabled)
        - Flip U/D (defaults to disabled)
        - Brightness (defaults to disabled)
        - Contrast (defaults to disabled)
        - Noise (off by default)
        - Motion blur (off by default)
- Model validation: This step will also be done through the Ultralytics library.
    - Validation will consist of getting back the performance graphs, a sample of model predictions, and the confusion matrix of the best weights found in training.

## How to Use This Notebook
### Running locally vs through Colab
This notebook is a combination of documentation and executable. It will not work through the Google Colab CLI or using ROCm, so in these cases you should use __model_trainer.py__.

For local training, create an isolated Python environment from the repository root. Use either `venv` or Conda, then select that environment as the notebook kernel. The dependency-install cell at the end of this notebook installs the training packages into the active kernel. Roboflow is only needed when downloading a dataset from Roboflow; Ultralytics installs PyTorch and torchvision as dependencies. For GPU training, install the PyTorch build appropriate for your operating system and CUDA/ROCm setup using the [PyTorch installation instructions](https://pytorch.org/get-started/locally/).

### Preparing to Run Locally
##### Option 1: Create a venv

Run these commands from the repository root:

```bash
cd PPE/training
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install ipykernel
python -m ipykernel install --user --name ppe-training --display-name "Python (ppe-training)"
```

On Windows, activate the environment with `.venv\Scripts\activate` instead of the `source` command. In VS Code or Jupyter, choose **Python (ppe-training)** as the notebook kernel.

##### Option 2: Create a Conda environment

Run these commands from the repository root:

```bash
cd PPE/training
conda create --name ppe-training python=3.12 -y
conda activate ppe-training
python -m pip install --upgrade pip ipykernel
python -m ipykernel install --user --name ppe-training --display-name "Python (ppe-training)"
```

Choose **Python (ppe-training)** as the notebook kernel after creating either environment. Then run the dependency-install cell below.

##### Quick caveat for ROCm and AMD GPUs
If using AMD GPUs, all GPU compute is routed through a tool called ROCm. This comes with some important caveats:
- ROCm and PyTorch for ROCm must be installed. Links [here](https://rocm.docs.amd.com/en/latest/install/rocm.html) and [here](https://rocm.docs.amd.com/projects/ai-ecosystem/en/latest/frameworks/pytorch/install.html).
- You should install all dependencies manually, not through requirements.txt, as requirements.txt assumes you are using CPU or CUDA.
- With ROCm, you can *ONLY* compute on a dataset stored *LOCALLY*. RoboFlow does not work with ROCm as the dependencies conflict.
- If using ROCm, use a virtual environment that does not contain the RoboFlow library.

- __Local__: If you are planning on running this locally on an NVidia GPU, you are uploading this to Google Colab to run it there, or you plan on running everything on the CPU and waiting for ages, you're already in the right place! Everything in this notebook should be decently interactive and user-friendly. You can also use model_trainer.py if you want 
- Alternatively, if you plan on running any of this through the Google Colab CLI, you should use __model_trainer.py__ and pass in these arguments:

#### Required arguments

- `--model`: YOLO model (`yolo11s.pt`, `yolo11n.pt`, `yolo26s.pt`, or `yolo26n.pt`).
- Choose one dataset source:
  - `--data`: Local dataset YAML file; use this for downloaded datasets. *IMPORTANT* This only works locally.
  - `--workspace`, `--dataset`, `--version`, and `--api-key`: Roboflow workspace, project, version, and API key.

#### Training arguments

- `--epochs`: Maximum training epochs (default: `300`).
- `--patience`: Early-stopping patience (default: `50`).
- `--imgsz`: Training image size (default: `640`).
- `--batch`: Images per batch (default: `16`; lower it if GPU memory is limited).
- `--device`: Training device, such as `0`, `cpu`, or `auto` (default: `auto`).
- `--workers`: Data-loader worker processes (default: `8`).
- `--project`: Output directory (default: `runs/train`).
- `--name`: Name for this training run (default: `train`).
- `--seed`: Random seed for repeatable runs (default: `42`).
- `--lr0`: Initial learning rate (default: `0.01`).

#### Optional augmentation arguments

Omit these flags to leave the corresponding augmentation disabled.

- `--rotate`: Maximum rotation in degrees.
- `--shear`: Maximum shear in degrees.
- `--mosaic`: Mosaic augmentation probability from `0` to `1`.
- `--flip-lr`: Left/right flip probability from `0` to `1`.
- `--flip-ud`: Up/down flip probability from `0` to `1`.
- `--brightness`: Brightness adjustment strength.
- `--contrast`: Contrast adjustment strength.
- `--noise`: Noise strength (off by default).
- `--motion-blur`: Motion-blur kernel size in pixels (default when enabled: `3`).


### Running on Colab
There are two ways to run this on Colab.
- If you want to run the full notebook, then you should take this file, upload it to Colab, and use the web interface, running each code cell in turn.
- If you would rather run the entire pipeline at once from the CLI, then you should run the Colab CLI alongside the __model_trainer.py__ script. Do not attempt to run this notebook under Colab CLI, as it will not work.

For example, run the local script on a Colab GPU and forward its arguments with `colab run`:

```bash
colab run --gpu T4 model_trainer.py \
  --model yolo11s.pt \
  --workspace YOUR_WORKSPACE \
  --dataset YOUR_DATASET \
  --version 1 \
  --api-key "$ROBOFLOW_API_KEY" \
  --epochs 300 \
  --patience 50 \
  --imgsz 640 \
  --batch 16 \
  --device auto \
  --workers 8 \
  --project runs/train \
  --name ppe-yolo11s \
  --seed 42 \
  --lr0 0.01
```

For a local dataset, replace the four Roboflow flags with `--data path/to/data.yaml`. Set `ROBOFLOW_API_KEY` in your local shell before running the command rather than writing the key directly in it.

## Part 1: Installing Dependencies

In [ ]:
# Runs a system check to ensure GPU access
!nvidia-smi

In [ ]:
%pip install ultralytics roboflow albumentations

## Part 2: Setting up directories and importing dataset from RoboFlow

First, we will import os, get the current working directory, and set that to HOME. This constant will then be used throughout this notebook to store the outputs of all commands.

In [1]:
import os
HOME = os.getcwd()
print(HOME)

/home/james/PPE-detection/PPE/training


Then, we will add in some imports and run Ultralytics self-checks:

In [2]:
import ultralytics
from IPython import display

display.clear_output()

ultralytics.checks()

Ultralytics 8.4.183 🚀 Python-3.12.15 torch-2.9.1+cu128 CPU (Intel Core i5-9600K 3.70GHz)
Setup complete ✅ (6 CPUs, 39.1 GB RAM, 392.8/456.9 GB disk)


Next, we will import display/image tools from IPython:

In [3]:
from IPython.display import display, Image
import ipywidgets as w
from pathlib import Path

Now it's time to grab the dataset from Roboflow. The next cell prompts for the workspace, project, version, model type, and API key using widgets that work in both a local notebook and Colab.

In [4]:
datasets_dir = Path(HOME) / "datasets"
datasets_dir.mkdir(parents=True, exist_ok=True)
os.chdir(datasets_dir)

We will then log into RoboFlow so we aren't passing around API keys all willy-nilly.

In [6]:
import roboflow

roboflow.login()

visit https://app.roboflow.com/auth-cli to get your authentication token.


This section will ask for the following information:
- Workspace (The RoboFlow workspace your data imports from)
- Project (The exact dataset project your data is from)
- Version (The version of that dataset, numbered)
- Model (The YOLO model version you will use, such as YOLOv11n)

To find the workspace, project, and version, you can go to the RoboFlow project, find the version of your dataset you wish to download, click "Download Dataset", and select the "Get Download Code" option. This option will show you Python code for downloading from RoboFlow, but within that code you will see strings for the workspace, project, and version.
![Export from RoboFlow dialog](public/export_from_roboflow.png)

In [ ]:
from getpass import getpass
from pathlib import Path

workspace = w.Text(
    description="Workspace:", 
    placeholder="my-workspace", 
    style={"description_width": "120px"}
)
project = w.Text(
    description="Project:",
    placeholder="my-project",
    style={"description_width": "120px"}
)
version = w.BoundedIntText(
    description="Version:", 
    value=1, 
    min=1, 
    max=9999, 
    style={"description_width": "120px"}
)
model = w.Dropdown(
    description="Model type:", 
    options=["yolo11s", "yolo11n", "yolo26s", "yolo26n"],
    value="yolo11s", 
    style={"description_width": "120px"}
)
run_btn = w.Button(description="Confirm", button_style="success")
out = w.Output()

dataset = None
config = {}

export_formats = {
    "yolo11s": "yolov11",
    "yolo11n": "yolov11",
    "yolo26s": "yolo26",
    "yolo26n": "yolo26",
}

def on_click(_):
    global dataset
    with out:
        out.clear_output()
        if not workspace.value.strip() or not project.value.strip():
            print("Workspace and project are required.")
            return
        config.update(
            workspace_name=workspace.value.strip(),
            project_name=project.value.strip(),
            version=version.value,
            model_type=model.value,
            export_type=export_formats.get(model.value),
        )
        if not config.get("workspace_name") or not config.get("project_name"):
            raise ValueError("Roboflow workspace and project are required.")

        try:
            version_number = int(config.get("version"))
        except ValueError as exc:
            raise ValueError("Dataset version must be a positive integer.") from exc
        if version_number < 1:
            raise ValueError("Dataset version must be a positive integer.")

        rf = roboflow.Roboflow()

        rf_project = rf.workspace(config.get("workspace_name")).project(config.get("project_name"))
        dataset = rf_project.version(version_number).download(config.get("export_type"))

run_btn.on_click(on_click)

display(w.VBox([workspace, project, version, model, run_btn, out]))

## Part 3: Training
In this section, a model will actually get trained based on a set of selected parameters. You will be given a set of inputs with which you can set up data augmentations, number of epochs, patience, and other important params.
- Epochs: How many cycles the training will go through
- Patience: How many epochs can occur with no tangible improvement before training stops early. 0 means no early stopping.
- Rotate: A number of degrees up to which rotation may happen.
- Shear: Number of degrees the image may be "slanted" from a normal rectangular shape
- Flip L/R and U/D: Probability of flipping left to right and up to down, respectively
- Brightness: Changes the brightness of the image
- Contrast: Changes contrast/exposure of the image
- Noise: Adds digital noise-like effects to the image at the intensity specified
- Blur: Adds n pixels of motion-blur

For more info on augmentations, see the [Ultralytics guide](https://docs.ultralytics.com/guides/yolo-data-augmentation/).

In [ ]:
style = {"description_width": "160px"}

epochs_w     = w.BoundedIntText(description="Epochs:", value=300, min=1, max=10000, style=style)
patience_w   = w.BoundedIntText(description="Patience:", value=50, min=0, max=10000, style=style)
imgsz_w      = w.BoundedIntText(description="Image size:", value=640, min=32, max=2048, step=32, style=style)
batch_w      = w.BoundedIntText(description="Batch size:", value=16, min=1, max=512, style=style)
rotate_w     = w.BoundedFloatText(description="Rotate (deg):", value=0, min=0, max=180, style=style)
shear_w      = w.BoundedFloatText(description="Shear (deg):", value=0, min=0, max=45, style=style)
mosaic_w     = w.FloatSlider(description="Mosaic prob:", value=0, min=0, max=1, step=0.05, style=style)
fliplr_w     = w.FloatSlider(description="Flip L/R prob:", value=0, min=0, max=1, step=0.05, style=style)
flipud_w     = w.FloatSlider(description="Flip U/D prob:", value=0, min=0, max=1, step=0.05, style=style)
brightness_w = w.FloatSlider(description="Brightness:", value=0, min=0, max=1, step=0.05, style=style)
contrast_w   = w.FloatSlider(description="Contrast:", value=0, min=0, max=1, step=0.05, style=style)
noise_w      = w.FloatSlider(description="Noise:", value=0, min=0, max=1, step=0.05, style=style)
blur_w       = w.BoundedIntText(description="Motion blur (px):", value=3, min=0, max=31, style=style)

display(w.VBox([epochs_w, patience_w, imgsz_w, batch_w, rotate_w, shear_w, mosaic_w,
                fliplr_w, flipud_w, brightness_w, contrast_w, noise_w, blur_w]))

In [ ]:
import torch
from ultralytics import YOLO
import albumentations as A

if dataset is None:
    raise RuntimeError("Download the dataset in Part 2 first (click Confirm).")

os.chdir(HOME)  # Part 2 chdir'd into datasets/, so weight downloads would land there otherwise
runs_dir = Path(HOME) / "runs" / "train"
runs_dir.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = 0
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

args = dict(
    data=str(Path(dataset.location) / "data.yaml"),
    epochs=epochs_w.value,
    patience=patience_w.value,
    imgsz=imgsz_w.value,
    batch=batch_w.value,
    device=device,
    project=str(runs_dir),
    name=f"{config['project_name']}-{config['model_type']}",
    degrees=rotate_w.value,
    shear=shear_w.value,
    mosaic=mosaic_w.value,
    fliplr=fliplr_w.value,
    flipud=flipud_w.value,
    hsv_v=brightness_w.value,
)

extra = []
if contrast_w.value:
    extra.append(A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=contrast_w.value, p=0.5))
if noise_w.value:
    extra.append(A.GaussNoise(std_range=(0, noise_w.value), p=0.5))
if blur_w.value:
    k = blur_w.value if blur_w.value % 2 else blur_w.value + 1
    extra.append(A.MotionBlur(blur_limit=(k, k), p=0.5))
args["augmentations"] = extra

yolo = YOLO(f"{config['model_type']}.pt")
yolo.train(**args)
print("Outputs in:", Path(yolo.trainer.save_dir))
print("Best weights:", Path(yolo.trainer.save_dir) / "weights" / "best.pt")

### Part 4: Validation
Here you'll see a few cells that you can use to validate the model. They'll display the stats for the model during its training, as well as some validation metrics.
- mAP: Mean absolute precision; roughly translates to true positives, higher is better
- Recall: Higher is better; high recall translates to fewer false positives
- Confusion Matrix: Measurement of how many of box i existed vs how many were measured

These results will show for the weights "best.pt" which are the weights during training at which the best performance was achieved.

In [ ]:
save_dir = Path(yolo.trainer.save_dir)

for name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    p = save_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p), width=900))

preds = sorted(save_dir.glob("val_batch*_pred.jpg"), key=lambda p: int("".join(filter(str.isdigit, p.stem))))
if preds:
    print(preds[-1].name)
    display(Image(filename=str(preds[-1]), width=900))

In [ ]:
save_dir = Path(yolo.trainer.save_dir)
best = YOLO(save_dir / "weights" / "best.pt")

metrics = best.val(
    data=str(Path(dataset.location) / "data.yaml"),
    imgsz=imgsz_w.value,
    batch=batch_w.value,
    device=device,
    project=str(runs_dir),
    name=f"{save_dir.name}-val",
)

print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
for c in metrics.box.ap_class_index:
    print(f"  {metrics.names[c]:<20} {metrics.box.maps[c]:.3f}")